In [1]:
# Load environment and suppress warnings
import os
import warnings
from dotenv import load_dotenv

load_dotenv()
warnings.filterwarnings("ignore")

api_key = os.getenv("OPENAI_API_KEY")
print("Key loaded:", bool(api_key))

if not api_key:
    raise ValueError("OPENAI_API_KEY not found.")

Key loaded: True


In [2]:
# Import required components and initialize deterministic model
from typing import Dict, Any
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from langchain_core.callbacks import BaseCallbackHandler

model = ChatOpenAI(model="gpt-5-mini", temperature=0)
parser = StrOutputParser()

In [3]:
# Define a sample request
# request_id is metadata used for tracing and logging
request = {
    "ticket_text": "Production checkout failing with 500 errors after deployment.",
    "request_id": "REQ-2049"
}

In [4]:
# Step 1: extract a short issue phrase
issue_prompt = ChatPromptTemplate.from_template(
    "Extract a short issue phrase: {ticket_text}"
)

issue_chain = issue_prompt | model | parser

def add_issue(ctx: Dict[str, Any]) -> Dict[str, Any]:
    updated = dict(ctx)
    updated["issue"] = issue_chain.invoke(
        {"ticket_text": ctx["ticket_text"]}
    ).strip()
    return updated

In [5]:
# Step 2: classify ticket priority
priority_prompt = ChatPromptTemplate.from_template(
    "Classify priority as exactly one word: High, Medium, or Low.\nTicket: {ticket_text}"
)

priority_chain = priority_prompt | model | parser

def add_priority(ctx: Dict[str, Any]) -> Dict[str, Any]:
    updated = dict(ctx)
    priority = priority_chain.invoke(
        {"ticket_text": ctx["ticket_text"]}
    ).strip().split()[0].replace(".", "")
    updated["priority"] = priority
    return updated

In [6]:
# Step 3: recommend a support action
action_prompt = ChatPromptTemplate.from_template(
    "Suggest one support action for issue: {issue}"
)

action_chain = action_prompt | model | parser

In [7]:
# Compose the workflow
workflow = (
    RunnableLambda(add_issue)
    | RunnableLambda(add_priority)
    | action_chain
)

# Run workflow without callbacks (baseline execution)
baseline = workflow.invoke(request)

print("=== FINAL OUTPUT (NO CALLBACKS) ===")
print(baseline)

=== FINAL OUTPUT (NO CALLBACKS) ===
Perform an immediate rollback to the last known-good release to restore checkout functionality — but first capture/ preserve current server logs, request traces, and the deployment artifact for post‑mortem analysis.


In [8]:
# Define a callback handler to trace execution

class ExecutionTraceCallback(BaseCallbackHandler):

    def on_chain_start(self, serialized, inputs, **kwargs):
        print("\n[TRACE] Chain started")

    def on_chain_end(self, outputs, **kwargs):
        print("[TRACE] Chain finished")

    def on_llm_start(self, serialized, prompts, **kwargs):
        print("[TRACE] LLM call started")

    def on_llm_end(self, response, **kwargs):
        print("[TRACE] LLM call finished")

In [9]:
# Run workflow with callbacks enabled
trace = ExecutionTraceCallback()

traced_output = workflow.invoke(
    request,
    config={
        "callbacks": [trace],
        "metadata": {"request_id": request["request_id"]}
    }
)

print("\n=== FINAL OUTPUT (WITH CALLBACKS) ===")
print(traced_output)


[TRACE] Chain started

[TRACE] Chain started

[TRACE] Chain started

[TRACE] Chain started
[TRACE] Chain finished
[TRACE] LLM call started
[TRACE] LLM call finished

[TRACE] Chain started
[TRACE] Chain finished
[TRACE] Chain finished
[TRACE] Chain finished

[TRACE] Chain started

[TRACE] Chain started

[TRACE] Chain started
[TRACE] Chain finished
[TRACE] LLM call started
[TRACE] LLM call finished

[TRACE] Chain started
[TRACE] Chain finished
[TRACE] Chain finished
[TRACE] Chain finished

[TRACE] Chain started
[TRACE] Chain finished
[TRACE] LLM call started
[TRACE] LLM call finished

[TRACE] Chain started
[TRACE] Chain finished
[TRACE] Chain finished

=== FINAL OUTPUT (WITH CALLBACKS) ===
Rollback the deployment to the last known-good release to restore checkout availability.

How to do it (high level):
- Identify the last successful deploy in your deployment tool (CI/CD, Kubernetes ReplicaSet, release history in Heroku/CapRover/etc.).
- Trigger a rollback/restore to that release (e.g.

In [10]:
# Inspect intermediate results step by step

after_issue = add_issue(request)
print("\n=== AFTER ISSUE EXTRACTION ===")
print({"issue": after_issue["issue"]})

after_priority = add_priority(after_issue)
print("\n=== AFTER PRIORITY CLASSIFICATION ===")
print({
    "issue": after_priority["issue"],
    "priority": after_priority["priority"]
})


=== AFTER ISSUE EXTRACTION ===
{'issue': 'Production checkout 500 errors after deployment'}

=== AFTER PRIORITY CLASSIFICATION ===
{'issue': 'Production checkout 500 errors after deployment', 'priority': 'High'}
